# Use nested-pandas data with Hyrax

This notebook shows how to read a parquet-backed `nested-pandas` `NestedFrame`
with `NestedPandasDataset`.

The important naming rule is that top-level columns keep their original names,
while nested subcolumns are exposed to Hyrax as `<nested_column>__<subcolumn>`.
For example, the nested-pandas column access `lightcurve.time` becomes the Hyrax
field name `lightcurve__time`.

In [ ]:
from pathlib import Path

import hyrax
import pandas as pd
from nested_pandas.nestedframe import NestedFrame

In [ ]:
flat_data = pd.DataFrame(
    {
        "object_id": ["obj_1", "obj_1", "obj_2", "obj_2", "obj_2"],
        "ra": [10.0, 10.0, 20.0, 20.0, 20.0],
        "dec": [-1.0, -1.0, 0.5, 0.5, 0.5],
        "time": [1.0, 2.0, 1.5, 2.5, 3.5],
        "flux": [100.0, 101.0, 50.0, 51.0, 52.0],
        "band": ["g", "r", "g", "r", "i"],
    }
)

nested_frame = NestedFrame.from_flat(
    flat_data,
    base_columns=["object_id", "ra", "dec"],
    nested_columns=["time", "flux", "band"],
    on="object_id",
    name="lightcurve",
)

data_path = Path("./nested_lightcurves.parquet")
nested_frame.to_parquet(data_path)
nested_frame

In [ ]:
h = hyrax.Hyrax()
h.config["data_set"]["NestedPandasDataset"]["read_parquet_kwargs"] = {}
h.config["data_request"] = {
    "train": {
        "science": {
            "dataset_class": "NestedPandasDataset",
            "data_location": str(data_path),
            "fields": ["object_id", "ra", "lightcurve__time", "lightcurve__flux"],
            "primary_id_field": "object_id",
            "split_fraction": 1.0,
        }
    }
}

prepared = h.prepare()
dataset = prepared["train"]._primary_or_first_dataset()
dataset.get_lightcurve__flux(0)